# Semantic Error Detection in Structured Extraction

Schema validation catches **Format Errors** (wrong format for existing data). It is blind to **Semantic Errors** — outputs that are format-valid but carry the wrong or fabricated meaning.

| Error type | What happened | Format validation catches it? | Fix |
|---|---|---|---|
| **Format Error** | Data exists; Claude used wrong format | ✅ regex / schema detects it | Retry with error message |
| **Semantic Error — Missing Information** | Field absent; `required` forced fabrication | ❌ fabricated value is format-valid | Nullable schema |
| **Semantic Error — Misinterpretation** | Field exists; Claude misread it | ❌ misread value is format-valid | Confidence scoring + human review |

This notebook focuses on the two Semantic Error subtypes and how to surface them:

| Stage | Concept | Key technique |
|---|---|---|
| 1. Misinterpretation baseline | Ambiguous source data silently produces wrong extractions | Show the failure mode |
| 2. Confidence scoring | Ask Claude to rate its certainty per field | Per-field confidence output |
| 3. Routing logic | Fields below threshold go to Human Review Queue | Threshold routing |

> **Related notebooks:**  
> Missing Information (Semantic Error subtype 1) → [`schema_validation_retry.ipynb`](schema_validation_retry.ipynb)  
> Nullable schema design → [`null_handling_and_normalization.ipynb`](null_handling_and_normalization.ipynb)

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## Stage 1: Misinterpretation — The Invisible Failure Mode

Misinterpretation happens when source data **exists** but is ambiguous — the model picks one reading and outputs a format-valid value that passes all validation checks. Unlike Missing Information, retrying does not help: the ambiguity is in the source document, not in Claude's formatting.

The invoice below contains two ambiguous fields:
- `issue_date`: `5/3/26` — could be May 3 (MM/DD) or March 5 (DD/MM)
- `payment_terms`: `"within thirty days"` — loosely implies Net 30 but is not stated explicitly

Running three independent extractions shows diverging results — the diagnostic signature of a Semantic Error.

In [ ]:
// Ambiguous invoice: issue_date and payment_terms can be read multiple ways
const ambiguousInvoice =
  'Invoice #INV-2026-0312. Issued 5/3/26. ' +
  'Payment due within thirty days. ' +
  'Amount due: $4,250. Handled by: R. Torres.';

const strictInvoiceTool: Anthropic.Tool = {
  name: 'extract_invoice',
  description: 'Extract key fields from an invoice.',
  input_schema: {
    type: 'object',
    properties: {
      invoice_number: { type: 'string', description: 'Invoice identifier' },
      issue_date:     { type: 'string', description: 'Issue date in YYYY-MM-DD' },
      amount:         { type: 'number', description: 'Total amount due in USD' },
      payment_terms:  { type: 'string', description: 'Payment terms, e.g. Net 30' },
    },
    required: ['invoice_number', 'issue_date', 'amount', 'payment_terms'],
  },
};

console.log('--- Ambiguous invoice: 3 independent extractions ---');
console.log('');

for (let i = 1; i <= 3; i++) {
  const response = await client.messages.create({
    model: MODEL,
    max_tokens: 512,
    tools: [strictInvoiceTool],
    tool_choice: { type: 'tool', name: 'extract_invoice' },
    messages: [{ role: 'user', content: ambiguousInvoice }],
  });
  const block = response.content.find(b => b.type === 'tool_use');
  if (block?.type === 'tool_use') {
    const data = block.input as Record<string, unknown>;
    console.log(
      `Attempt ${i}:  issue_date=${data['issue_date']}  payment_terms=${data['payment_terms']}`
    );
  }
}

console.log('');
console.log('Diagnosis: different values across attempts → Misinterpretation Semantic Error.');
console.log('The data exists in the source, so retrying will not converge.');
console.log('All outputs are format-valid — schema validation cannot catch this.');

## Stage 2: Confidence Scoring

The defense against Misinterpretation is to ask Claude to attach a **confidence score** (0.0–1.0) to each extracted field. When the source is ambiguous, the model's uncertainty should surface as a lower score.

The schema wraps each field in an object with two properties:
- `value` — the extracted value (nullable for fields that may be absent)
- `confidence` — a number from 0.0 (no basis) to 1.0 (certain)

The system prompt instructs Claude to lower confidence when a field is ambiguous or could be interpreted multiple ways.

In [ ]:
const confidenceInvoiceTool: Anthropic.Tool = {
  name: 'extract_invoice_with_confidence',
  description: 'Extract invoice fields with a confidence score (0.0–1.0) for each field.',
  input_schema: {
    type: 'object',
    properties: {
      invoice_number: {
        type: 'object',
        properties: {
          value:      { type: 'string' },
          confidence: { type: 'number', description: '0.0 (no basis) to 1.0 (certain)' },
        },
        required: ['value', 'confidence'],
      },
      issue_date: {
        type: 'object',
        properties: {
          value:      { type: ['string', 'null'], description: 'YYYY-MM-DD, or null if unreadable' },
          confidence: { type: 'number' },
        },
        required: ['value', 'confidence'],
      },
      amount: {
        type: 'object',
        properties: {
          value:      { type: 'number' },
          confidence: { type: 'number' },
        },
        required: ['value', 'confidence'],
      },
      payment_terms: {
        type: 'object',
        properties: {
          value:      { type: ['string', 'null'], description: 'e.g. Net 30, or null if not stated' },
          confidence: { type: 'number' },
        },
        required: ['value', 'confidence'],
      },
    },
    required: ['invoice_number', 'issue_date', 'amount', 'payment_terms'],
  },
};

const CONFIDENCE_SYSTEM =
  'For each field, report your confidence honestly. ' +
  'If a field is ambiguous or could be interpreted multiple ways, lower your confidence score. ' +
  'If a field is absent, return null for value and a low confidence score.';

const confidenceResponse = await client.messages.create({
  model: MODEL,
  max_tokens: 1024,
  system: CONFIDENCE_SYSTEM,
  tools: [confidenceInvoiceTool],
  tool_choice: { type: 'tool', name: 'extract_invoice_with_confidence' },
  messages: [{ role: 'user', content: ambiguousInvoice }],
});

const confidenceBlock = confidenceResponse.content.find(b => b.type === 'tool_use');
if (confidenceBlock?.type === 'tool_use') {
  console.log('Extraction with per-field confidence scores:');
  console.log(JSON.stringify(confidenceBlock.input, null, 2));
  console.log('');
  console.log('Observe: invoice_number and amount are explicit → high confidence.');
  console.log('issue_date is ambiguous (5/3/26) → lower confidence.');
  console.log('payment_terms is inferred ("thirty days") → lower confidence.');
}

## Stage 3: Routing by Confidence Threshold

The confidence score becomes a **routing signal**:

```
Each field's confidence score
          │
  ┌───────┴───────┐
≥ threshold      < threshold
  │                    │
Automated          Human Review Queue
Downstream         (flag for manual check)
Processing
```

The threshold is not fixed — it depends on the business context:

| Domain | Threshold | Reason |
|---|---|---|
| Financial compliance | 0.99 | Error cost is high; prefer more human review |
| Internal reporting | 0.80 | Error cost is low; minimize human cost |
| Medical records | 0.99+ | Regulatory requirements |

Human reviewers see **only low-confidence fields**, not the full document — keeping the review workload minimal.

In [ ]:
const THRESHOLD = 0.85;

type FieldResult = { value: unknown; confidence: number };
type ExtractionResult = Record<string, FieldResult>;

function routeByConfidence(
  data: ExtractionResult,
  threshold: number,
): { automated: Record<string, unknown>; needsReview: Record<string, FieldResult> } {
  const automated: Record<string, unknown> = {};
  const needsReview: Record<string, FieldResult> = {};

  for (const [field, result] of Object.entries(data)) {
    if (result.confidence >= threshold) {
      automated[field] = result.value;
    } else {
      needsReview[field] = result;
    }
  }

  return { automated, needsReview };
}

if (confidenceBlock?.type === 'tool_use') {
  const { automated, needsReview } = routeByConfidence(
    confidenceBlock.input as ExtractionResult,
    THRESHOLD,
  );

  console.log(`Routing threshold: ${THRESHOLD * 100}%`);
  console.log('');
  console.log('→ Automated Downstream Processing (confidence ≥ threshold):');
  console.log(JSON.stringify(automated, null, 2));
  console.log('');
  console.log('→ Human Review Queue (confidence < threshold):');
  console.log(JSON.stringify(needsReview, null, 2));
  console.log('');
  const reviewCount = Object.keys(needsReview).length;
  const totalCount = Object.keys(confidenceBlock.input as object).length;
  console.log(`Human reviewer sees ${reviewCount}/${totalCount} fields — not the full document.`);
}

## Summary

### Semantic Error: two subtypes

| Subtype | Root cause | Detection | Fix |
|---|---|---|---|
| **Missing Information** | Field absent; `required` forces fabrication | Diverging values on retry | Nullable schema (remove from `required`, use `[type, null]`) |
| **Misinterpretation** | Field exists but source is ambiguous | Diverging values on retry | Confidence scoring + human review |

Both subtypes produce **format-valid output** — schema validation and retry loops cannot distinguish them from correct results.

### Diagnosis: retry divergence is the shared signal

| Retry pattern | Diagnosis | Action |
|---|---|---|
| Converges in 1-2 attempts | Format Error | Retry worked |
| Same wrong value every time | Format Error (unresolved) | Improve error message |
| Different value every attempt | **Semantic Error** (either subtype) | Do not retry — fix source |

Once you identify a Semantic Error via divergence, determine which subtype:
- Field **absent** from source → Missing Information → fix schema
- Field **present but ambiguous** → Misinterpretation → add confidence scoring

### Three-layer extraction defense

```
Layer 1: Nullable schema
  → Prevents Missing Information (fabrication under required pressure)
  → null is honest absence; downstream can branch on === null

Layer 2: Validation retry loop
  → Fixes Format Errors (wrong format for existing data)
  → Surfaces Semantic Errors via divergence diagnosis

Layer 3: Confidence scoring + threshold routing
  → Detects Misinterpretation (ambiguous source data)
  → Routes low-confidence fields to Human Review Queue
```

Each layer catches what the others miss. A production extraction pipeline needs all three.

### Related notebooks

- Retry loop + divergence diagnosis → [`schema_validation_retry.ipynb`](schema_validation_retry.ipynb)
- Nullable schema design + format normalization → [`null_handling_and_normalization.ipynb`](null_handling_and_normalization.ipynb)